In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import polars as pl
import datetime as dt

import numpy as np
from pomap_v1.categorical_pomap import CategoricalPoMap
from pomap_v1.pomap import PoMap

ModuleNotFoundError: No module named 'pomap_v1'

In [3]:
example_df = pl.from_dict(
    {'cat1': ['a',  'b',],}
)

timestamps = pl.datetime_range(dt.datetime(2025, 1, 1),
                               dt.datetime(2025, 1, 7),
                               interval='1h',
                               eager=True
                               ).rename('ts').to_frame()

example_df = example_df.join(timestamps, how='cross')
example_df = example_df.with_columns(
    feature = np.random.normal(0, 1, example_df.shape[0])
)

example_df = example_df.with_columns(hour=pl.col('ts').dt.hour())

In [18]:
pmap_cat = CategoricalPoMap(dims_to_labels={'cat1': ['a', 'b']})

In [19]:
# Fetch the training data corresponding to this label
pmap_cat.label_to_train(df=example_df,
                        label={'cat1': 'a'}
                        ).unique('cat1').select('cat1')

cat1
str
"""a"""


In [20]:
example_df.select('hour').unique().to_dict()

{'hour': shape: (24,)
 Series: 'hour' [i8]
 [
 	6
 	12
 	9
 	13
 	2
 	…
 	20
 	18
 	23
 	1
 	19
 ]}

In [21]:
# Form a Two dimensional PoMap
dims_to_labels = {}
for dim in ['hour', 'cat1']:
    dims_to_labels[dim] = example_df[dim].unique().to_list()

pmap_both = CategoricalPoMap(dims_to_labels=dims_to_labels)

# Check that it's giving us the right train data
pmap_both.label_to_train(example_df, {'cat1': 'a', 'hour': 6}).unique(['cat1', 'hour']).select('cat1', 'hour')

cat1,hour
str,i8
"""a""",6


In [22]:
# Now test the composition
pmap_hour = CategoricalPoMap({'hour': dims_to_labels['hour']})
pmap_composed = PoMap.product(pmap_hour, pmap_cat)
pmap_composed.label_to_train(example_df, {'cat1': 'a', 'hour': 6}).unique(['cat1', 'hour']).select('cat1', 'hour')

AttributeError: property 'labels' of 'PoMap' object has no setter

In [29]:
# Test inverting the order of the composition
pmap_composed = PoMap.product(pmap_cat, pmap_hour)
pmap_composed.label_rows_as_test(example_df, {'cat1': 'a', 'hour': 6}).unique(['cat1', 'hour']).select('cat1', 'hour')

cat1,hour
str,i8
"""b""",7
"""a""",10
"""b""",20
"""b""",19
"""a""",6
…,…
"""a""",8
"""b""",3
"""a""",0


# Build a Model

In [ ]:
# Here's the original function we want to lift

# lightgbm.train(params,
#                train_set: lgbm.Dataset,
#                num_boost_round=100,
#                valid_sets=None,
#                valid_names=None,
#                feval=None,
#                init_model=None,
#                keep_training_booster=False,
#                callbacks=None
#                )

# Specific params will be provided by PoMap, in this case
# train_set, valid_sets, and plausibly valid_names.

# We would still like to retain all the other kwargs!
# Then it behaves exactly the same as the original function.
# This is easily solved with **kwargs.

In [ ]:
# Let's just start by writing out long-form, then figuring out the abstraction

def lifted_fit(
        model_constructor,
        train_df: pl.DataFrame, # Must be Polars, by the restriction of PoMap,
        pomap: PoMap,
        target_column,
        **train_kwargs,
):
    # We need to provide a list of labels, it's natural that PoMap does this.
    
    models = {}
    
    for label in pomap.labels:
        
        sub_model = model_constructor()
        
        sub_train = pomap.label_to_train(train_df, label)
        sub_validate = pomap.label_to_validate(train_df, label)
        
        sub_train_target = sub_train.select(target_column).to_pandas()
        sub_validate_target = sub_validate.select(target_column).to_pandas()
        
        sub_train = sub_train.drop(target_column).to_pandas()
        sub_validate = sub_validate.drop(target_column).to_pandas()
        
        sub_model = sub_model.fit(X=sub_train,
                                  y=sub_train_target,
                                  eval_set=[(sub_validate, sub_validate_target)],
                                  **train_kwargs
                                  )
        
        models[label] = sub_model
    
    return models

In [ ]:
# Can we convert this to a decorator which provides the arguments for the function?
# Idea is, you feed it a function, and it converts to a copy of the function which:
    # Will iterate through the labels.
    # Get the model for that label.
    # Get the data for the proposed argument using pomap.
    # Run the function with the arguments using the cached data 
    # Allocate the data into a specified return structure.
    # Return the specified structure.

# Honestly, feels quite hard to deal with abstractly because the interfaces
# And types expected by models are all different.

# Let's just ignore it and do it long form - most models don't have that many methods that we actually need to lift (I think)


In [ ]:
class LiftedLgbm:
    
    def __init__(self, lgbm_constructor, pomap: PoMap, target_column):
        self.pomap = pomap
        self.target_column = target_column
        self.models = {
            label: lgbm_constructor() for label in pomap.labels
        }        
    
    def fit(self, df: pl.DataFrame, target_column, **fit_kwargs):
        
        for label in self.pomap.labels:            
            sub_train = self.pomap.label_to_train(df, label)
            sub_validate = self.pomap.label_to_validate(df, label)
            
            sub_train_target = sub_train.select(target_column).to_pandas()
            sub_validate_target = sub_validate.select(target_column).to_pandas()
            
            sub_train = sub_train.drop(target_column).to_pandas()
            sub_validate = sub_validate.drop(target_column).to_pandas()
            
            self.models[label] = self.models[label].fit(
                X=sub_train,
                y=sub_train_target,
                eval_set=[(sub_validate, sub_validate_target)],
                **fit_kwargs
              )
            
        return 
    
    def predict(self, df: pl.DataFrame):
        df = df.with_row_index(name='_index')
        
        outs = []
        for label in self.pomap.labels:
            df = self.pomap.label_rows_as_test(df, label)
            sub_df = df.filter(self.pomap.train_column_name(label) == label)
            predictions = self.models[label].predict(
                sub_df.drop('_index', self.target_column)
                .to_pandas()
            )
            sub_df = sub_df.with_columns(prediction=predictions)
            outs.append(sub_df)
        out = pl.concat(df).select('_index', 'prediction')
        df = df.join(out, on='_index').drop('_index')
    
        return df